# Airline Arrival Delay Prediction

This notebook focuses only on **feature selection and prediction** using the cleaned airline dataset.

Goal: Predict whether a flight will arrive late using schedule-based features.

## 1. Start Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AirlineArrivalDelayPrediction") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark session started.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 13:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started.


26/04/28 13:59:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Load Cleaned Dataset

The cleaned data is stored in the Spark output folder `cleaned_airline_data`.

In [2]:
df = spark.read.csv(
    "cleaned_airline_data",
    header=True,
    inferSchema=True
)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

[Stage 2:>                                                        (0 + 10) / 11]

Rows: 7070784
Columns: 42
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- DepDelayMinutes: double (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: d

## 3. Feature Engineering

We add two simple schedule-based features:

- `is_peak_hour`: whether the flight departs during morning/evening peak hours
- `is_weekend`: whether the flight is scheduled on a weekend

In [3]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "is_peak_hour",
    when((col("dep_hour") >= 6) & (col("dep_hour") <= 10), 1)
    .when((col("dep_hour") >= 16) & (col("dep_hour") <= 20), 1)
    .otherwise(0)
)

df = df.withColumn(
    "is_weekend",
    when(col("DayOfWeek").isin([6, 7]), 1).otherwise(0)
)

df.select("dep_hour", "DayOfWeek", "is_peak_hour", "is_weekend").show(10)

+--------+---------+------------+----------+
|dep_hour|DayOfWeek|is_peak_hour|is_weekend|
+--------+---------+------------+----------+
|       8|        1|           1|         0|
|       8|        2|           1|         0|
|       8|        3|           1|         0|
|       8|        4|           1|         0|
|       8|        5|           1|         0|
|       8|        1|           1|         0|
|       8|        2|           1|         0|
|       8|        3|           1|         0|
|       8|        4|           1|         0|
|       8|        5|           1|         0|
+--------+---------+------------+----------+
only showing top 10 rows



## 4. Feature Selection

We avoid leakage columns such as actual arrival delay, actual departure delay, arrival time, departure time, and delay-cause columns because these values are not available before the flight happens.

Target variable:

- `is_arr_delayed`

In [4]:
model_df = df.select(
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour",
    "season",
    "is_peak_hour",
    "is_weekend",
    "is_arr_delayed"
).dropna()

print("Modeling rows:", model_df.count())
print("Modeling columns:", len(model_df.columns))
model_df.show(5)

[Stage 6:==========================================>               (8 + 3) / 11]

Modeling rows: 7070783
Modeling columns: 14
+-----+----------+---------+-----------------+------+----+----------+--------------+--------+--------+------+------------+----------+--------------+
|Month|DayofMonth|DayOfWeek|Reporting_Airline|Origin|Dest|CRSDepTime|CRSElapsedTime|Distance|dep_hour|season|is_peak_hour|is_weekend|is_arr_delayed|
+-----+----------+---------+-----------------+------+----+----------+--------------+--------+--------+------+------------+----------+--------------+
|    1|         8|        1|               9E|   LGA| OMA|       856|         219.0|  1148.0|       8|Winter|           1|         0|             0|
|    1|         9|        2|               9E|   LGA| OMA|       856|         219.0|  1148.0|       8|Winter|           1|         0|             0|
|    1|        10|        3|               9E|   LGA| OMA|       856|         219.0|  1148.0|       8|Winter|           1|         0|             0|
|    1|        11|        4|               9E|   LGA| OMA|    

## 5. Optional Sampling for Local Machines

The full dataset has millions of rows. If Spark crashes on your laptop, use a sample.

Leave this cell commented if the full dataset runs successfully.

In [ ]:
# Uncomment this if your laptop crashes or Spark runs out of memory.
# model_df = model_df.sample(fraction=0.30, seed=42)
# print("Sampled rows:", model_df.count())

## 6. Prepare Features for Machine Learning

Categorical columns are first indexed and then one-hot encoded.  
One-hot encoding is better for Logistic Regression because it avoids treating categories as ordered numbers.

In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

categorical_cols = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "season"
]

numeric_cols = [
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "CRSDepTime",
    "CRSElapsedTime",
    "Distance",
    "dep_hour",
    "is_peak_hour",
    "is_weekend"
]

label_col = "is_arr_delayed"

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categorical_cols],
    outputCols=[c + "_vec" for c in categorical_cols]
)

feature_cols = [c + "_vec" for c in categorical_cols] + numeric_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

## 7. Train-Test Split

In [6]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 5656679


[Stage 13:==================================================>     (10 + 1) / 11]

Testing rows: 1414104


## 8. Train Logistic Regression Model

Logistic Regression is used because it is scalable, simple, and provides probability estimates for arrival delay.

In [7]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.2
)

pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])

lr_model = pipeline.fit(train_df)

predictions = lr_model.transform(test_df)

print("Model training completed.")

26/04/28 14:03:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

Model training completed.


## 9. Evaluate Model

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

precision = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

recall = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions)

print("===== Logistic Regression Results =====")
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Precision:", precision)
print("Recall:", recall)
print("AUC:", auc)

[Stage 64:==================================================>     (10 + 1) / 11]

===== Logistic Regression Results =====
Accuracy: 0.797256071689211
F1 Score: 0.7073196233499506
Precision: 0.6356172438453123
Recall: 0.797256071689211
AUC: 0.632353333694543


## 10. Confusion Matrix

In [ ]:
conf_matrix = predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction")
conf_matrix.show()

## 11. Sample Predictions

The `probability` column shows the estimated probability for each class:

- class 0: not delayed
- class 1: delayed

In [ ]:
predictions.select(
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "DayOfWeek",
    "CRSDepTime",
    "Distance",
    "is_arr_delayed",
    "prediction",
    "probability"
).show(20, truncate=False)

## 12. Save Predictions

This saves selected prediction outputs into a Spark folder called `arrival_delay_predictions`.

In [ ]:
predictions.select(
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "DayOfWeek",
    "CRSDepTime",
    "Distance",
    "is_arr_delayed",
    "prediction",
    "probability"
).write.mode("overwrite").option("header", True).csv("arrival_delay_predictions")

print("Predictions saved to folder: arrival_delay_predictions")

## 13. Final Modeling Summary

This notebook built a Spark ML pipeline to predict arrival delays using schedule-based features.  
The model avoids leakage columns and uses one-hot encoding for categorical variables before training Logistic Regression.

In [ ]:
print("""
Final Summary:
A Logistic Regression model was trained using Spark MLlib to predict arrival delays.

Target:
- is_arr_delayed

Features:
- airline
- origin airport
- destination airport
- month
- day of month
- day of week
- scheduled departure time
- scheduled elapsed time
- distance
- departure hour
- season
- peak-hour indicator
- weekend indicator

Leakage columns such as actual arrival delay and delay-cause variables were excluded.
""")